# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Publication Date: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview

Review available record sets, fields, their `@id`s, and related columns. Each entity can be referenced by its `@id` within `mlcroissant`.

In [ ]:
# List all available record sets and their fields
print("Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")
    fields_str = ', '.join([f.name+f' (@id: {f.id})' for f in rs.fields])
    print(f"  Fields: {fields_str if fields_str else 'No fields listed'}")
    # Columns info
    for f in rs.fields:
        if hasattr(f, 'columns') and f.columns:
            print(f"    Field '{f.name}' columns: {', '.join([c.name+' (@id: '+c.id+')' for c in f.columns])}")

## 3. Data Extraction

Select one or more record sets by their `@id` and extract their data as Pandas DataFrames. Use `@id`s for all references (`record_set`, `fields`, etc.).

In [ ]:
# Gather all record set IDs (using their @id)
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Use mlcroissant's generator to load all records in this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set '{record_set_id}' with {len(df)} rows and {len(df.columns)} columns.")

# For demonstration, select the first record set for further analysis
if len(record_sets) > 0:
    main_rs_id = record_sets[0]
    print(f"\nColumns in '{main_rs_id}':\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No RecordSets found in the schema.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. We use fields referenced by their `@id` for consistency.

In [ ]:
# Example EDA on numeric field (by @id)
# You must set these IDs based on the field overview above or by investigating df.columns

# Replace with the actual record set @id you want to analyze
record_set_id = main_rs_id  # use the main record set loaded above
df = dataframes[record_set_id]

# Try to automatically pick a numeric field for demo; otherwise, set manually
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
    # Drop NA for robust analysis
    filtered_df = df[df[numeric_field_id] > 0]  # simple threshold filter (replace as appropriate)
    print(f"Filtered records with {numeric_field_id} > 0:")
    print(filtered_df.head())

    # Z-score normalization of numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by first object/categorical column
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize a numeric field's distribution and relationships with another field, where available, using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data or numeric field found to visualize.')

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load Croissant metadata and records using `mlcroissant` by schema URL
- Explore record sets, fields, and their `@id` for robust, reproducible code
- Extract and analyze the structured data
- Perform basic exploratory data analysis on the extracted data
- Visualize distributions and relationships

Adapt this notebook with additional analyses as needed. Always reference data entities by their `@id` for clarity and consistency.